# Stage 6-1. CNN 아키텍처 — im2col, 레이어, 파라미터 비교

이 노트북은 `src/nn/conv.py`와 `src/models/cnn.py`를 실습한다.

| 클래스/함수 | 역할 |
|---|---|
| `im2col(x, K, stride, padding, xp)` | (B, C, H, W) 입력을 (B·out_h·out_w, C·K·K) 행렬로 변환 |
| `col2im(col, x_shape, K, stride, padding, xp)` | im2col 역변환 — gradient 전파에 사용 |
| `Conv2d(in_c, out_c, K, stride, padding, seed, xp)` | 2D 합성곱 레이어 |
| `MaxPool2d(K, stride, padding, xp)` | 2D 최대 풀링 레이어 |
| `Flatten()` | (B, C, H, W) → (B, C·H·W) 변환 |
| `Dropout(p)` | 학습 시 랜덤 마스크, 평가 시 통과 |
| `CNN(task, seed)` | im2col 기반 CNN 모델 — MLP와 동일한 외부 인터페이스 |

**학습 목표**
1. im2col이 합성곱을 행렬곱으로 변환하는 원리를 이해한다.
2. Conv2d, MaxPool2d, Flatten, Dropout 각 레이어의 shape 흐름을 추적한다.
3. CNN 모델 구조와 MLP 대비 파라미터 수를 비교한다.
4. CuPy/NumPy 경계 처리 방식을 확인한다.
5. `train()` / `eval()` 모드 전환과 Dropout 동작을 검증한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import cupy as xp
    _BACKEND = "CuPy"
except ImportError:
    import numpy as xp
    _BACKEND = "NumPy (CuPy 미설치 — fallback)"

print(f"백엔드: {_BACKEND}")

from src.nn.conv import im2col, col2im, Conv2d, MaxPool2d, Flatten, Dropout
from src.nn.layers import Linear, ReLU, Sequential
from src.models.cnn import CNN
from src.models.mlp import MLP

## 1. im2col — 합성곱을 행렬곱으로

### 1.1. im2col 원리

합성곱 연산은 커널을 입력 위에서 슬라이딩하며 원소별 곱셈-합산을 반복한다.
`im2col`은 이 반복을 **하나의 행렬곱**으로 바꿔 GPU/CPU 행렬 연산을 최대한 활용한다.

```
입력 (B, C, H, W)
  ↓  im2col(K=3, stride=1, padding=0)
col (B·out_h·out_w, C·K·K)   ← 각 행이 커널 윈도우 하나
  ↓  @ weight_matrix (out_c, C·K·K).T
출력 (B·out_h·out_w, out_c)
  ↓  reshape + transpose
(B, out_c, out_h, out_w)
```

`out_h = (H + 2·pad - K) // stride + 1`  
`out_w = (W + 2·pad - K) // stride + 1`

In [ ]:
# 소형 입력으로 im2col shape 확인
B, C, H, W = 2, 1, 4, 4
K, stride, padding = 3, 1, 0

x = np.arange(B * C * H * W, dtype=np.float32).reshape(B, C, H, W)
x_xp = xp.asarray(x)

col, out_h, out_w = im2col(x_xp, K, stride, padding, xp=xp)

print("입력 shape:", x_xp.shape, f"→ (B={B}, C={C}, H={H}, W={W})")
print(f"커널: K={K}, stride={stride}, padding={padding}")
print(f"출력 공간: out_h={out_h}, out_w={out_w}")
print(f"col shape: {col.shape}")
print(f"  = (B·out_h·out_w, C·K·K) = ({B}·{out_h}·{out_w}, {C}·{K}·{K}) = ({B*out_h*out_w}, {C*K*K})")

In [ ]:
# padding 효과 확인
col_pad, out_h_p, out_w_p = im2col(x_xp, K, stride, padding=1, xp=xp)

print("padding=0 →", f"out=({out_h}, {out_w})", f"col shape={col.shape}")
print("padding=1 →", f"out=({out_h_p}, {out_w_p})", f"col shape={col_pad.shape}")
print()
print("패딩 효과: 동일한 커널 크기에서 입력과 출력 공간 크기가 같아진다 (same padding)")

### 1.2. im2col → 행렬곱 → 합성곱 결과

In [ ]:
# im2col로 직접 합성곱 수행
B, C, H, W = 1, 1, 5, 5
out_c, K = 2, 3

x = np.random.default_rng(0).standard_normal((B, C, H, W)).astype(np.float32)
x_xp = xp.asarray(x)

# im2col 변환
col, out_h, out_w = im2col(x_xp, K, stride=1, padding=0, xp=xp)

# 무작위 커널 (out_c, C, K, K)
rng = np.random.default_rng(42)
w_np = rng.standard_normal((out_c, C, K, K)).astype(np.float32)
w_xp = xp.asarray(w_np)
b_xp = xp.zeros(out_c, dtype=np.float32)

# 행렬곱으로 합성곱 수행
col_w = w_xp.reshape(out_c, -1)          # (out_c, C*K*K)
result = col @ col_w.T + b_xp            # (B*out_h*out_w, out_c)
result = result.reshape(B, out_h, out_w, out_c).transpose(0, 3, 1, 2)  # (B, out_c, out_h, out_w)

result_np = result.get() if hasattr(result, "get") else np.asarray(result)
print(f"입력 (B, C, H, W): {x.shape}")
print(f"커널 (out_c, C, K, K): {w_np.shape}")
print(f"출력 (B, out_c, out_h, out_w): {result_np.shape}")

### 1.3. im2col / col2im 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("im2col 원리 — (B=1, C=1, H=4, W=4), K=3, stride=1, padding=0",
             fontsize=12, fontweight="bold")

# 입력 이미지
x_vis = np.arange(16, dtype=np.float32).reshape(4, 4)
ax = axes[0]
im = ax.imshow(x_vis, cmap="Blues", vmin=0, vmax=15)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{int(x_vis[i, j])}", ha="center", va="center", fontsize=12, fontweight="bold")
ax.set_title("입력 (4×4)", fontsize=11)
ax.set_xticks([])
ax.set_yticks([])
# 첫 번째 커널 윈도우 표시
rect = plt.Rectangle((-0.5, -0.5), 3, 3, linewidth=2, edgecolor="red", facecolor="none")
ax.add_patch(rect)
ax.text(1, 1.5, "윈도우 0", ha="center", va="center", color="red", fontsize=9)

# col 행렬
x_xp_vis = xp.asarray(x_vis.reshape(1, 1, 4, 4))
col_vis, oh, ow = im2col(x_xp_vis, 3, 1, 0, xp=xp)
col_np = col_vis.get() if hasattr(col_vis, "get") else np.asarray(col_vis)

ax = axes[1]
im2 = ax.imshow(col_np, cmap="Blues", aspect="auto", vmin=0, vmax=15)
for i in range(col_np.shape[0]):
    for j in range(col_np.shape[1]):
        ax.text(j, i, f"{int(col_np[i, j])}", ha="center", va="center", fontsize=9)
ax.set_title(f"im2col 결과 ({col_np.shape[0]}×{col_np.shape[1]})\n각 행=윈도우, 각 열=커널 원소", fontsize=10)
ax.set_xlabel("C·K·K = 9")
ax.set_ylabel("B·out_h·out_w = 4")

# shape 흐름 도식
ax = axes[2]
ax.axis("off")
steps = [
    ("입력\n(B, C, H, W)", "(1, 1, 4, 4)", "#AED6F1"),
    ("im2col", "↓", "white"),
    ("col\n(B·oH·oW, C·K·K)", "(4, 9)", "#A9DFBF"),
    ("@ weight.T + b", "↓", "white"),
    ("출력 행렬\n(B·oH·oW, out_c)", "(4, out_c)", "#F9E79F"),
    ("reshape + transpose", "↓", "white"),
    ("출력\n(B, out_c, oH, oW)", "(1, out_c, 2, 2)", "#F5CBA7"),
]
for k, (label, val, color) in enumerate(steps):
    y = 1.0 - k * 0.145
    if val == "↓":
        ax.text(0.5, y, val, ha="center", va="center", fontsize=16, color="gray",
                transform=ax.transAxes)
    else:
        ax.text(0.5, y, f"{label}\n{val}", ha="center", va="center", fontsize=10,
                transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.3", facecolor=color, edgecolor="gray"))
ax.set_title("shape 흐름", fontsize=11)

plt.tight_layout()
plt.show()

## 2. Conv2d — forward / backward shape

In [ ]:
# Conv2d: 1채널 → 32채널, 3×3 커널, same padding
conv1 = Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1, seed=0, xp=xp)

x_in = xp.asarray(np.random.randn(8, 1, 28, 28).astype(np.float32))
out = conv1.forward(x_in)

print("[ Conv2d(1→32, K=3, pad=1) ]")
print(f"  입력:   {x_in.shape}")
print(f"  출력:   {out.shape}")
print(f"  w:      {conv1.w.shape}   ← (out_c, in_c, K, K)")
print(f"  b:      {conv1.b.shape}   ← (out_c,)")
param_count = conv1.w.size + conv1.b.size
print(f"  파라미터 수: {param_count:,}  (w={conv1.w.size:,} + b={conv1.b.size:,})")

In [ ]:
# backward shape 확인
dout = xp.ones_like(out)
dx = conv1.backward(dout)

print("[ Conv2d backward ]")
print(f"  dout:   {dout.shape}")
print(f"  dx:     {dx.shape}   ← 입력과 같은 shape")
print(f"  grad_w: {conv1.grad_w.shape}")
print(f"  grad_b: {conv1.grad_b.shape}")

## 3. MaxPool2d — shape 추적

In [ ]:
pool = MaxPool2d(kernel_size=2, stride=2, xp=xp)

# Conv2d 출력을 MaxPool 입력으로 사용
pool_in = out   # (8, 32, 28, 28)
pool_out = pool.forward(pool_in)

print("[ MaxPool2d(K=2, stride=2) ]")
print(f"  입력:  {pool_in.shape}")
print(f"  출력:  {pool_out.shape}")
print(f"  공간 크기: 28×28 → 14×14 (1/2 다운샘플링)")

# backward
dpool = xp.ones_like(pool_out)
dpool_in = pool.backward(dpool)
print(f"\n[ MaxPool2d backward ]")
print(f"  dout:  {dpool.shape}")
print(f"  dx:    {dpool_in.shape}   ← 입력과 같은 shape")

## 4. CNN 전체 shape 흐름

In [ ]:
# CNN 내부 각 단계 shape을 수동으로 추적
from src.nn.conv import Conv2d as _Conv2d, MaxPool2d as _MaxPool2d, Flatten as _Flatten
from src.nn.layers import Linear as _Linear, ReLU as _ReLU

B = 4
x_flat = np.random.randn(B, 784).astype(np.float32)  # MnistDataset 출력 형식

print("[ CNN shape 흐름 추적 (task=multiclass, B=4) ]")
print()

# --- forward 단계별 ---
stages = []

x0 = xp.asarray(x_flat).reshape(B, 1, 28, 28)
stages.append(("입력 reshape", "(N, 784) → (N, 1, 28, 28)", x0.shape))

conv1_ = _Conv2d(1, 32, 3, padding=1, seed=0, xp=xp)
x1 = conv1_.forward(x0)
stages.append(("Conv2d(1→32, K=3, pad=1)", "same padding: 28 유지", x1.shape))

r1 = _ReLU()
x2 = r1.forward(x1)
stages.append(("ReLU", "shape 변화 없음", x2.shape))

p1 = _MaxPool2d(2, 2, xp=xp)
x3 = p1.forward(x2)
stages.append(("MaxPool2d(K=2, stride=2)", "28 → 14", x3.shape))

conv2_ = _Conv2d(32, 64, 3, padding=1, seed=1, xp=xp)
x4 = conv2_.forward(x3)
stages.append(("Conv2d(32→64, K=3, pad=1)", "same padding: 14 유지", x4.shape))

r2 = _ReLU()
x5 = r2.forward(x4)
stages.append(("ReLU", "shape 변화 없음", x5.shape))

p2 = _MaxPool2d(2, 2, xp=xp)
x6 = p2.forward(x5)
stages.append(("MaxPool2d(K=2, stride=2)", "14 → 7", x6.shape))

fl = _Flatten()
x7_xp = fl.forward(x6)
x7 = x7_xp.get() if hasattr(x7_xp, "get") else np.asarray(x7_xp)
stages.append(("Flatten", "64×7×7 = 3136 (CuPy→numpy 변환)", x7.shape))

fc1 = _Linear(3136, 256, seed=2)
x8 = fc1.forward(x7)
stages.append(("Linear(3136→256)", "", x8.shape))

r3 = _ReLU()
x9 = r3.forward(x8)
stages.append(("ReLU", "shape 변화 없음", x9.shape))

fc2 = _Linear(256, 10, seed=3)
x10 = fc2.forward(x9)
stages.append(("Linear(256→10)", "output_dim=10 (multiclass)", x10.shape))

print(f"{'레이어':<35} {'비고':<30} {'출력 shape'}")
print("-" * 80)
for name, note, shape in stages:
    print(f"{name:<35} {note:<30} {str(tuple(shape))}")

In [ ]:
# shape 흐름 시각화
layer_names = [s[0] for s in stages]
shapes      = [s[2] for s in stages]

# 총 원소 수 (B 제외)
def feature_size(shape):
    return int(np.prod(shape[1:]))

sizes = [feature_size(s) for s in shapes]

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["#AED6F1"] * 3 + ["#A9DFBF"] * 2 + ["#AED6F1"] * 2 + ["#F9E79F"] * 3 + ["#F5CBA7"]
bars = ax.bar(range(len(sizes)), sizes, color=colors[:len(sizes)], edgecolor="gray", alpha=0.85)

for i, (bar, sz, shape) in enumerate(zip(bars, sizes, shapes)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f"{sz:,}", ha="center", va="bottom", fontsize=8, rotation=45)
    ax.text(bar.get_x() + bar.get_width()/2, -150,
            "×".join(str(d) for d in shape[1:]), ha="center", va="top", fontsize=7, color="#555")

ax.set_xticks(range(len(layer_names)))
ax.set_xticklabels([s[0] for s in stages], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Feature 원소 수 (batch 제외)")
ax.set_title("CNN shape 흐름 — feature 크기 변화", fontsize=12, fontweight="bold")
ax.set_ylim(-300, max(sizes) * 1.25)

legend_patches = [
    mpatches.Patch(color="#AED6F1", label="Conv 경로 (CuPy)"),
    mpatches.Patch(color="#A9DFBF", label="Pooling"),
    mpatches.Patch(color="#F9E79F", label="FC 경로 (NumPy)"),
    mpatches.Patch(color="#F5CBA7", label="출력 logit"),
]
ax.legend(handles=legend_patches, loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. CNN 파라미터 수 — MLP 비교

In [ ]:
cnn = CNN(task="multiclass", seed=42)
mlp = MLP(task="multiclass", seed=42)

def count_params(model):
    return sum(p.size for p in model.params)

def param_breakdown(model, name):
    print(f"\n[ {name} ]")
    for i, p in enumerate(model.params):
        p_np = p.get() if hasattr(p, "get") else p
        label = "w" if i % 2 == 0 else "b"
        layer_idx = i // 2
        print(f"  params[{i}] (layer{layer_idx} {label}): shape={p_np.shape}, size={p_np.size:,}")
    print(f"  총 파라미터: {count_params(model):,}")

param_breakdown(cnn, "CNN")
param_breakdown(mlp, "MLP")

In [ ]:
# 파라미터 수 시각화
def get_param_table(model, name):
    rows = []
    for i, p in enumerate(model.params):
        p_np = p.get() if hasattr(p, "get") else p
        label = "w" if i % 2 == 0 else "b"
        layer_idx = i // 2
        rows.append((f"layer{layer_idx}_{label}", p_np.shape, p_np.size))
    return rows

cnn_table = get_param_table(cnn, "CNN")
mlp_table = get_param_table(mlp, "MLP")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("CNN vs MLP — 레이어별 파라미터 수", fontsize=13, fontweight="bold")

for ax, table, model_name in zip(axes, [cnn_table, mlp_table], ["CNN", "MLP"]):
    labels = [r[0] for r in table]
    sizes_  = [r[2] for r in table]
    shapes_ = [str(r[1]) for r in table]
    total   = sum(sizes_)

    cmap = plt.cm.Set2
    bar_colors = [cmap(i % 8) for i in range(len(labels))]
    bars = ax.bar(labels, sizes_, color=bar_colors, edgecolor="gray", alpha=0.85)
    for bar, sz, sh in zip(bars, sizes_, shapes_):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total * 0.005,
                f"{sz:,}", ha="center", va="bottom", fontsize=8)
        ax.text(bar.get_x() + bar.get_width()/2, -total * 0.02,
                sh, ha="center", va="top", fontsize=7, color="#555")

    ax.set_title(f"{model_name} (총 {total:,}개)", fontsize=11)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("파라미터 수")
    ax.set_ylim(-total * 0.06, total * 1.18)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

cnn_total = count_params(cnn)
mlp_total = count_params(mlp)
print(f"CNN 총 파라미터: {cnn_total:,}")
print(f"MLP 총 파라미터: {mlp_total:,}")
print(f"CNN/MLP 비율:   {cnn_total/mlp_total:.2f}x")

## 6. Dropout — train / eval 모드

In [ ]:
drop = Dropout(p=0.5)
np.random.seed(42)

x_test = np.ones((1, 1000), dtype=np.float32)

# train 모드 (기본)
drop.train()
out_train = drop.forward(x_test)
active_ratio = (out_train != 0).mean()

# eval 모드
drop.eval()
out_eval = drop.forward(x_test)

print("[ Dropout(p=0.5) ]")
print(f"  train 모드 — 활성 뉴런 비율: {active_ratio:.3f} (기댓값: 0.5)")
print(f"  train 모드 — 활성 뉴런 평균값: {out_train[out_train != 0].mean():.3f}")
print(f"  (inverted dropout: 활성 뉴런을 1/(1-p)={1/(1-0.5):.1f}배 스케일 업)")
print()
print(f"  eval 모드  — 출력 평균: {out_eval.mean():.3f} (모두 1.0 통과)")
print(f"  eval 모드  — 입력 == 출력: {np.array_equal(x_test, out_eval)}")

In [ ]:
# CNN train/eval 모드 전파 확인
cnn = CNN(task="multiclass", seed=42)

x_sample = np.random.randn(4, 784).astype(np.float32)

# eval 모드: 동일 입력 → 동일 출력
cnn.eval()
out_eval_1 = cnn.forward(x_sample)
out_eval_2 = cnn.forward(x_sample)

# train 모드: Dropout이 매번 다른 마스크 사용
cnn.train()
out_train_1 = cnn.forward(x_sample)
out_train_2 = cnn.forward(x_sample)

print("[ CNN train/eval 모드 ]")
print(f"  eval 모드 결정론적 (동일 출력): {np.allclose(out_eval_1, out_eval_2)}")
print(f"  train 모드 확률적 (다른 출력):  {not np.allclose(out_train_1, out_train_2)}")
print()
print(f"  cnn.dropout.training: {cnn.dropout.training}  ← train() 호출 후")
cnn.eval()
print(f"  cnn.dropout.training: {cnn.dropout.training}  ← eval() 호출 후")

## 7. CuPy/NumPy 경계 처리

In [ ]:
# CNN forward에서 CuPy/NumPy 변환이 일어나는 지점 확인
import types

cnn = CNN(task="multiclass", seed=42)
cnn.eval()

x_np = np.random.randn(2, 784).astype(np.float32)

# 단계별 배열 타입 추적
x_xp_in = cnn._xp.asarray(x_np).reshape(-1, 1, 28, 28)
after_conv = cnn.conv_net(x_xp_in)
after_flat_xp = cnn.flatten(after_conv)
after_flat_np = after_flat_xp.get() if hasattr(after_flat_xp, "get") else np.asarray(after_flat_xp)
logits = cnn.fc_net(after_flat_np)

def arr_type(a):
    return type(a).__module__.split(".")[0]  # "cupy" or "numpy"

print("[ CNN forward — 배열 타입 변환 지점 ]")
print(f"  입력 (MnistDataset 출력)    : {arr_type(x_np):>6} — shape={x_np.shape}")
print(f"  reshape to (N,1,28,28)       : {arr_type(x_xp_in):>6} — shape={x_xp_in.shape}")
print(f"  conv_net 통과 후             : {arr_type(after_conv):>6} — shape={after_conv.shape}")
print(f"  Flatten 직후 (변환 전)       : {arr_type(after_flat_xp):>6} — shape={after_flat_xp.shape}")
print(f"  Flatten 직후 (변환 후) ★    : {arr_type(after_flat_np):>6} — shape={after_flat_np.shape}")
print(f"  fc_net 통과 후 (logits)      : {arr_type(logits):>6} — shape={logits.shape}")
print()
print("  CuPy → NumPy 변환은 Flatten 직후 단 한 번만 발생")
print("  DataLoader / Trainer / 손실 함수는 수정 불필요")

## 8. 정리

**im2col 핵심 원리**
```
합성곱 = im2col → 행렬곱 → reshape
(B, C, H, W)  →  col (B·oH·oW, C·K·K)  @  w (out_c, C·K·K).T  →  (B, out_c, oH, oW)
```

**CNN 구조 요약**

| 레이어 | 입력 shape | 출력 shape | 파라미터 |
|---|---|---|---|
| Conv2d(1→32, K=3, pad=1) | (N, 1, 28, 28) | (N, 32, 28, 28) | 320 |
| MaxPool2d(2, 2) | (N, 32, 28, 28) | (N, 32, 14, 14) | 0 |
| Conv2d(32→64, K=3, pad=1) | (N, 32, 14, 14) | (N, 64, 14, 14) | 18,496 |
| MaxPool2d(2, 2) | (N, 64, 14, 14) | (N, 64, 7, 7) | 0 |
| Flatten | (N, 64, 7, 7) | (N, 3136) | 0 |
| Linear(3136→256) | (N, 3136) | (N, 256) | 803,072 |
| Linear(256→10) | (N, 256) | (N, 10) | 2,570 |

**CuPy/NumPy 경계**
- Conv/Pool 경로: CuPy 배열로 실행
- Flatten 직후: `x.get()` 으로 NumPy 변환 (단 1회)
- FC/Dropout 경로: NumPy 배열로 실행
- 외부 인터페이스: 입력과 출력 모두 NumPy → DataLoader, Trainer 수정 불필요

**Dropout 동작**
- `train()` 모드: 랜덤 마스크 생성, 활성 뉴런을 `1/(1-p)`배 스케일 업 (inverted dropout)
- `eval()` 모드: 마스크 없이 입력을 그대로 통과